# 01 — Exploration du jeu de départ AlphaForge

Chargement, conversion CAD et alignement des calendriers (exigences du README des données).

In [ ]:
import pandas as pd
import numpy as np

import os
from pathlib import Path

def find_data_dir():
    """Localise data/ même si le dossier de travail du noyau est invalide ou différent."""
    try:
        starts = [Path.cwd().resolve()]
    except FileNotFoundError:          # cwd supprimé : on repart d'un dossier valide
        os.chdir(Path.home())
        starts = []
    for start in starts:
        for base in [start, *start.parents]:
            for cand in (base / 'data', base):
                if (cand / 'prices_daily.csv').exists():
                    return cand
    # Repli : recherche du projet sous le home et sous /workshop
    for root in (Path.home(), Path('/workshop')):
        for f in root.rglob('alphaforge-ai/data/prices_daily.csv'):
            return f.parent
    raise FileNotFoundError("data/prices_daily.csv introuvable : définissez DATA à la main.")

DATA = find_data_dir()
print('DATA =', DATA)
universe = pd.read_csv(DATA / 'assets_universe.csv')
prices = pd.read_csv(DATA / 'prices_daily.csv', parse_dates=['date'])
fx = pd.read_csv(DATA / 'fx_usdcad_daily.csv', parse_dates=['date'])
macro = pd.read_csv(DATA / 'macro_rates.csv', parse_dates=['date'])
prices.head()

## Prix ajustés en format large
On utilise `adj_close` (ajusté dividendes et fractionnements) pour les rendements.

In [ ]:
wide = prices.pivot(index='date', columns='ticker', values='adj_close').sort_index()
wide.shape

## Conversion en CAD
Les tickers `.TO` sont en CAD, les autres en USD. Le fichier FX n'a pas de lignes les jours fériés canadiens : on reporte le dernier taux connu (hypothèse documentée).

In [ ]:
usdcad = fx.set_index('date')['usdcad'].reindex(wide.index).ffill()
usd_cols = [c for c in wide.columns if not c.endswith('.TO')]
wide_cad = wide.copy()
wide_cad[usd_cols] = wide_cad[usd_cols].mul(usdcad, axis=0)

## Alignement des calendriers
Hypothèse : on garde les jours ouvrables où le TSX et les marchés américains cotent tous deux ; la crypto est échantillonnée ces jours-là.

In [ ]:
equity_cols = [c for c in wide_cad.columns if not c.endswith('-USD')]
aligned = wide_cad.dropna(subset=equity_cols, how='any')
print(len(wide_cad), '->', len(aligned), 'jours après alignement')

## Indicateurs descriptifs (pas des prévisions)

In [ ]:
rets = aligned.pct_change().dropna(how='all')
stats = pd.DataFrame({
    'rendement_annuel': (aligned.iloc[-1] / aligned.iloc[0]) ** (252 / len(aligned)) - 1,
    'volatilite': rets.std() * np.sqrt(252),
    'pire_baisse': (aligned / aligned.cummax() - 1).min(),
}).join(universe.set_index('ticker')[['name', 'asset_class']])
stats.sort_values('volatilite').round(3)

## FNB sectoriels (XLK, XLE, XLF, XLV)
Téléchargement via yfinance, même période et même format que `prices_daily.csv`.

In [ ]:
import yfinance as yf

# FNB sectoriels, même période que le jeu de départ
tickers = ["XLK", "XLE", "XLF", "XLV"]
raw = yf.download(tickers, start="2021-11-12", end="2026-11-13",
                  auto_adjust=False, progress=False)

# Même format que prices_daily.csv
sect = (raw.stack(level=1, future_stack=True).reset_index()
           .rename(columns={"Date": "date", "Ticker": "ticker", "Open": "open",
                            "High": "high", "Low": "low", "Close": "close",
                            "Adj Close": "adj_close", "Volume": "volume"}))
sect = sect[["date", "ticker", "open", "high", "low", "close", "adj_close", "volume"]]
sect["currency"] = "USD"
sect.to_csv(DATA / "sector_etfs_daily.csv", index=False)
sect.head()